# Hey Banco — Datathon 2026
## Carga, inspección y joins de los datasets de transacciones

**Datasets:** `hey_clientes.csv` · `hey_productos.csv` · `hey_transacciones.csv`  
**Relaciones:** `hey_clientes.user_id` → `hey_productos.user_id` → `hey_transacciones.user_id`  
**Join secundario:** `hey_productos.producto_id` → `hey_transacciones.producto_id`

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:,.2f}'.format)

## 1. Carga de CSVs

In [2]:
BASE = Path(r"Datathon_Hey_2026_dataset_transacciones 1/dataset_transacciones")

clientes = pd.read_csv(
    BASE / "hey_clientes.csv",
    dtype={
        "user_id": str,
        "genero": str,
        "estado": str,
        "ciudad": str,
        "nivel_educativo": str,
        "ocupacion": str,
        "canal_apertura": str,
        "preferencia_canal": str,
        "idioma_preferido": str,
    },
)

productos = pd.read_csv(
    BASE / "hey_productos.csv",
    dtype={"producto_id": str, "user_id": str, "tipo_producto": str, "estatus": str},
    parse_dates=["fecha_apertura", "fecha_ultimo_movimiento"],
)

transacciones = pd.read_csv(
    BASE / "hey_transacciones.csv",
    dtype={
        "transaccion_id": str,
        "user_id": str,
        "producto_id": str,
        "tipo_operacion": str,
        "canal": str,
        "categoria_mcc": str,
        "ciudad_transaccion": str,
        "estatus": str,
        "motivo_no_procesada": str,
        "dia_semana": str,
        "dispositivo": str,
    },
    parse_dates=["fecha_hora"],
)

print("Datasets cargados correctamente.")

Datasets cargados correctamente.


## 2. Shapes

In [3]:
shapes = pd.DataFrame({
    "Dataset": ["hey_clientes", "hey_productos", "hey_transacciones"],
    "Filas": [clientes.shape[0], productos.shape[0], transacciones.shape[0]],
    "Columnas": [clientes.shape[1], productos.shape[1], transacciones.shape[1]],
})
display(shapes)

,Dataset,Filas,Columnas
0,hey_clientes,15025,22
1,hey_productos,38909,13
2,hey_transacciones,802384,22


## 3. Tipos de datos

In [4]:
print("=== hey_clientes ===")
display(clientes.dtypes.rename("dtype").to_frame())

print("\n=== hey_productos ===")
display(productos.dtypes.rename("dtype").to_frame())

print("\n=== hey_transacciones ===")
display(transacciones.dtypes.rename("dtype").to_frame())

=== hey_clientes ===


,dtype
user_id,object
edad,int64
sexo,object
estado,object
ciudad,object
nivel_educativo,object
ocupacion,object
ingreso_mensual_mxn,int64
antiguedad_dias,int64
es_hey_pro,bool



=== hey_productos ===


,dtype
producto_id,object
user_id,object
tipo_producto,object
fecha_apertura,datetime64[ns]
estatus,object
limite_credito,float64
saldo_actual,float64
utilizacion_pct,float64
tasa_interes_anual,float64
plazo_meses,float64



=== hey_transacciones ===


,dtype
transaccion_id,object
user_id,object
producto_id,object
fecha_hora,datetime64[ns]
tipo_operacion,object
canal,object
monto,float64
comercio_nombre,object
categoria_mcc,object
ciudad_transaccion,object


## 4. Valores nulos

In [5]:
def resumen_nulos(df: pd.DataFrame, nombre: str) -> pd.DataFrame:
    total = df.isnull().sum()
    pct = (total / len(df) * 100).round(2)
    resumen = pd.DataFrame({"nulos": total, "pct_%": pct})
    resumen = resumen[resumen["nulos"] > 0].sort_values("pct_%", ascending=False)
    print(f"=== {nombre}: {len(resumen)} columnas con nulos ===")
    return resumen

display(resumen_nulos(clientes, "hey_clientes"))
display(resumen_nulos(productos, "hey_productos"))
display(resumen_nulos(transacciones, "hey_transacciones"))

=== hey_clientes: 3 columnas con nulos ===


,nulos,pct_%
satisfaccion_1_10,751,5.00
estado,432,2.88
ciudad,432,2.88


=== hey_productos: 6 columnas con nulos ===


,nulos,pct_%
plazo_meses,34359,88.31
monto_mensualidad,34359,88.31
limite_credito,24592,63.20
utilizacion_pct,24592,63.20
tasa_interes_anual,20118,51.71
saldo_actual,3750,9.64


=== hey_transacciones: 7 columnas con nulos ===


,nulos,pct_%
meses_diferidos,785071,97.84
motivo_no_procesada,775775,96.68
cashback_generado,621746,77.49
comercio_nombre,370538,46.18
dispositivo,202063,25.18
descripcion_libre,61752,7.70
ciudad_transaccion,19267,2.40


## 5. Estadísticas descriptivas básicas

In [6]:
print("=== hey_clientes — columnas numéricas ===")
display(clientes.describe())

print("\n=== hey_clientes — columnas categóricas ===")
cat_cols_clientes = clientes.select_dtypes(include=["object", "bool"]).columns.tolist()
for col in cat_cols_clientes:
    vc = clientes[col].value_counts(dropna=False)
    print(f"\n{col} ({len(vc)} valores únicos):")
    display(vc.head(10))

=== hey_clientes — columnas numéricas ===


,edad,ingreso_mensual_mxn,antiguedad_dias,score_buro,dias_desde_ultimo_login,satisfaccion_1_10,num_productos_activos
count,"15,025.00","15,025.00","15,025.00","15,025.00","15,025.00","14,274.00","15,025.00"
mean,37.19,"29,793.68",935.11,618.45,18.77,7.48,2.23
std,10.19,"19,171.16",512.96,127.16,35.31,1.88,1.14
min,18.00,"4,500.00",7.00,295.00,0.00,3.00,0.00
25%,29.00,"16,000.00",490.00,529.00,2.00,6.00,1.00
50%,36.00,"24,000.00",942.00,631.00,7.00,8.00,2.00
75%,45.00,"37,000.00","1,375.00",719.00,15.00,9.00,3.00
max,60.00,"99,000.00","1,825.00",850.00,180.00,10.00,5.00



=== hey_clientes — columnas categóricas ===

user_id (15025 valores únicos):


user_id
USR-00001    1
USR-10022    1
USR-10010    1
USR-10011    1
USR-10012    1
USR-10013    1
USR-10014    1
USR-10015    1
USR-10016    1
USR-10017    1
Name: count, dtype: int64


sexo (3 valores únicos):


...

idioma_preferido
es_MX    14544
en_US      481
Name: count, dtype: int64


tiene_seguro (2 valores únicos):


tiene_seguro
False    10883
True      4142
Name: count, dtype: int64


patron_uso_atipico (2 valores únicos):


patron_uso_atipico
False    14262
True       763
Name: count, dtype: int64

In [7]:
print("=== hey_productos — columnas numéricas ===")
display(productos.describe())

print("\n=== hey_productos — tipo_producto ===")
display(productos["tipo_producto"].value_counts())

print("\n=== hey_productos — estatus ===")
display(productos["estatus"].value_counts())

=== hey_productos — columnas numéricas ===


,fecha_apertura,limite_credito,saldo_actual,utilizacion_pct,tasa_interes_anual,plazo_meses,monto_mensualidad,fecha_ultimo_movimiento
count,38909,"14,317.00","35,159.00","14,317.00","18,791.00","4,550.00","4,550.00",38909
mean,2024-08-17 04:30:16.793029632,"89,630.93","60,007.06",0.43,29.12,23.07,"1,990.70",2025-10-13 19:58:53.012927232
min,2020-12-21 00:00:00,"1,000.00",100.94,0.05,3.50,6.00,36.29,2025-08-30 00:00:00
25%,2023-12-17 00:00:00,"29,000.00","15,596.24",0.24,14.66,12.00,738.24,2025-09-21 00:00:00
50%,2024-12-07 00:00:00,"64,000.00","36,290.09",0.37,31.74,18.00,"1,477.32",2025-10-14 00:00:00
75%,2025-07-15 00:00:00,"111,000.00","66,193.26",0.58,41.06,24.00,"2,784.11",2025-11-06 00:00:00
max,2025-11-27 00:00:00,"494,000.00","499,905.57",1.00,66.97,60.00,"11,379.42",2025-11-28 00:00:00
std,NaN,"94,668.50","84,090.73",0.25,15.48,13.75,"1,683.13",NaN



=== hey_productos — tipo_producto ===


tipo_producto
cuenta_debito                  15025
tarjeta_credito_hey             7565
inversion_hey                   4474
seguro_vida                     2480
credito_nomina                  2044
credito_personal                1549
cuenta_negocios                 1343
seguro_compras                  1270
tarjeta_credito_negocios        1111
tarjeta_credito_garantizada     1091
credito_auto                     957
Name: count, dtype: int64


=== hey_productos — estatus ===


estatus
activo               33548
cancelado             2917
suspendido            1426
revision_de_pagos     1018
Name: count, dtype: int64

In [8]:
print("=== hey_transacciones — columnas numéricas ===")
display(transacciones.describe())

print("\n=== hey_transacciones — tipo_operacion ===")
display(transacciones["tipo_operacion"].value_counts())

print("\n=== hey_transacciones — estatus ===")
display(transacciones["estatus"].value_counts())

print("\n=== hey_transacciones — rango de fechas ===")
print(f"Desde: {transacciones['fecha_hora'].min()}")
print(f"Hasta: {transacciones['fecha_hora'].max()}")

=== hey_transacciones — columnas numéricas ===


,fecha_hora,monto,intento_numero,meses_diferidos,cashback_generado,hora_del_dia
count,802384,"802,384.00","802,384.00","17,313.00","180,638.00","802,384.00"
mean,2025-06-18 10:12:00.894490112,"6,108.50",1.03,10.78,14.79,11.44
min,2025-01-01 08:00:09,5.01,1.00,3.00,0.12,0.00
25%,2025-03-29 02:42:13.500000,540.00,1.00,6.00,3.56,6.00
50%,2025-06-18 01:01:14.500000,"1,740.00",1.00,9.00,6.93,11.00
75%,2025-09-07 19:14:19.750000128,"7,030.94",1.00,12.00,17.24,17.00
max,2025-12-15 13:59:36,"79,511.30",3.00,24.00,764.29,23.00
std,NaN,"9,885.31",0.23,7.20,24.63,6.76



=== hey_transacciones — tipo_operacion ===


tipo_operacion
compra               319524
transf_entrada        92167
transf_salida         90116
cargo_recurrente      67249
pago_credito          51651
pago_servicio         48562
abono_inversion       41769
retiro_cajero         38989
deposito_oxxo         23152
cashback              14226
retiro_inversion      11822
deposito_farmacia      3157
Name: count, dtype: int64


=== hey_transacciones — estatus ===


estatus
completada      748267
no_procesada     26609
en_disputa       19033
revertida         8475
Name: count, dtype: int64


=== hey_transacciones — rango de fechas ===
Desde: 2025-01-01 08:00:09
Hasta: 2025-12-15 13:59:36


## 6. Verificación de integridad referencial

In [9]:
ids_clientes = set(clientes["user_id"])
ids_clientes_en_productos = set(productos["user_id"])
ids_clientes_en_txn = set(transacciones["user_id"])

ids_productos = set(productos["producto_id"])
ids_productos_en_txn = set(transacciones["producto_id"])

print("--- user_id ---")
print(f"Clientes únicos en hey_clientes:          {len(ids_clientes):>8,}")
print(f"Clientes únicos en hey_productos:         {len(ids_clientes_en_productos):>8,}")
print(f"Clientes únicos en hey_transacciones:     {len(ids_clientes_en_txn):>8,}")
print(f"user_id en productos NO en clientes:      {len(ids_clientes_en_productos - ids_clientes):>8,}")
print(f"user_id en transacciones NO en clientes:  {len(ids_clientes_en_txn - ids_clientes):>8,}")

print("\n--- producto_id ---")
print(f"Productos únicos en hey_productos:        {len(ids_productos):>8,}")
print(f"Productos únicos en hey_transacciones:    {len(ids_productos_en_txn):>8,}")
print(f"producto_id en txn NO en productos:       {len(ids_productos_en_txn - ids_productos):>8,}")

--- user_id ---
Clientes únicos en hey_clientes:            15,025
Clientes únicos en hey_productos:           15,025
Clientes únicos en hey_transacciones:       15,025
user_id en productos NO en clientes:             0
user_id en transacciones NO en clientes:         0

--- producto_id ---
Productos únicos en hey_productos:          38,909
Productos únicos en hey_transacciones:      29,972
producto_id en txn NO en productos:              0


## 7. Joins

### 7a. Clientes + Productos (left join por `user_id`)

In [10]:
clientes_productos = clientes.merge(
    productos,
    on="user_id",
    how="left",
    suffixes=("_cli", "_prod"),
)

print(f"Shape: {clientes_productos.shape}")
print(f"Filas por cliente (media): {clientes_productos.groupby('user_id').size().mean():.2f}")
display(clientes_productos.head(3))

Shape: (38909, 34)
Filas por cliente (media): 2.59


,user_id,edad,sexo,estado,ciudad,nivel_educativo,ocupacion,ingreso_mensual_mxn,antiguedad_dias,es_hey_pro,nomina_domiciliada,canal_apertura,score_buro,dias_desde_ultimo_login,preferencia_canal,satisfaccion_1_10,recibe_remesas,usa_hey_shop,idioma_preferido,tiene_seguro,num_productos_activos,patron_uso_atipico,producto_id,tipo_producto,fecha_apertura,estatus,limite_credito,saldo_actual,utilizacion_pct,tasa_interes_anual,plazo_meses,monto_mensualidad,fecha_ultimo_movimiento,es_dato_sintetico
0,USR-00001,21,M,Ciudad de México,CDMX - Benito Juárez,Preparatoria,Empleado,24500,1554,True,False,App,527,1,app_android,10.00,False,True,es_MX,False,2,False,PRD-00000001,cuenta_debito,2023-06-26,activo,NaN,"80,954.60",NaN,NaN,NaN,NaN,2025-11-27,True
1,USR-00001,21,M,Ciudad de México,CDMX - Benito Juárez,Preparatoria,Empleado,24500,1554,True,False,App,527,1,app_android,10.00,False,True,es_MX,False,2,False,PRD-00000002,tarjeta_credito_hey,2022-10-16,activo,"144,000.00","88,790.40",0.62,35.71,NaN,NaN,2025-09-17,True
2,USR-00002,18,M,Jalisco,Puerto Vallarta,Preparatoria,Estudiante,19000,1410,True,False,App,714,3,app_android,8.00,False,True,es_MX,True,2,False,PRD-00000003,cuenta_debito,2025-04-03,activo,NaN,"20,712.54",NaN,NaN,NaN,NaN,2025-10-16,True


### 7b. Transacciones + Productos (inner join por `producto_id`)

In [11]:
txn_productos = transacciones.merge(
    productos[["producto_id", "user_id", "tipo_producto", "estatus",
               "limite_credito", "saldo_actual", "utilizacion_pct"]],
    on="producto_id",
    how="inner",
    suffixes=("_txn", "_prod"),
)

print(f"Shape: {txn_productos.shape}")
display(txn_productos.head(3))

Shape: (802384, 28)


,transaccion_id,user_id_txn,producto_id,fecha_hora,tipo_operacion,canal,monto,comercio_nombre,categoria_mcc,ciudad_transaccion,estatus_txn,motivo_no_procesada,intento_numero,meses_diferidos,cashback_generado,descripcion_libre,hora_del_dia,dia_semana,es_internacional,dispositivo,patron_uso_atipico,es_dato_sintetico,user_id_prod,tipo_producto,estatus_prod,limite_credito,saldo_actual,utilizacion_pct
0,TXN-0000000055,USR-00001,PRD-00000002,2025-01-15 14:17:42,compra,app_ios,33.88,DivertidoPark,entretenimiento,"Nueva York, NY",completada,NaN,1,NaN,0.34,Cargo automático,14,Wednesday,True,app_ios,False,True,USR-00001,tarjeta_credito_hey,activo,"144,000.00","88,790.40",0.62
1,TXN-0000000048,USR-00001,PRD-00000001,2025-01-17 00:31:56,cargo_recurrente,app_ios,249.00,GamerPass,servicios_digitales,CDMX - Benito Juárez,completada,NaN,1,NaN,NaN,Cargo automático,0,Friday,False,app_ios,False,True,USR-00001,cuenta_debito,activo,NaN,"80,954.60",NaN
2,TXN-0000000018,USR-00001,PRD-00000001,2025-01-17 22:48:23,cargo_recurrente,app_huawei,399.00,CloudDrive MX,servicios_digitales,CDMX - Benito Juárez,completada,NaN,1,NaN,NaN,inv hey,22,Friday,False,app_huawei,False,True,USR-00001,cuenta_debito,activo,NaN,"80,954.60",NaN


### 7c. Join completo: Clientes + Productos + Transacciones

In [12]:
df_full = transacciones.merge(
    productos[
        ["producto_id", "user_id", "tipo_producto", "estatus",
         "limite_credito", "saldo_actual", "utilizacion_pct",
         "tasa_interes_anual"]
    ],
    on="producto_id",
    how="left",
    suffixes=("", "_prod"),
).merge(
    clientes.drop(columns=[c for c in clientes.columns
                            if c.endswith("_sintetico")], errors="ignore"),
    on="user_id",
    how="left",
    suffixes=("_txn", "_cli"),
)

print(f"Shape join completo: {df_full.shape}")
print(f"Columnas ({df_full.shape[1]}):")
print(df_full.columns.tolist())
display(df_full.head(3))

Shape join completo: (802384, 50)
Columnas (50):
['transaccion_id', 'user_id', 'producto_id', 'fecha_hora', 'tipo_operacion', 'canal', 'monto', 'comercio_nombre', 'categoria_mcc', 'ciudad_transaccion', 'estatus', 'motivo_no_procesada', 'intento_numero', 'meses_diferidos', 'cashback_generado', 'descripcion_libre', 'hora_del_dia', 'dia_semana', 'es_internacional', 'dispositivo', 'patron_uso_atipico_txn', 'es_dato_sintetico', 'user_id_prod', 'tipo_producto', 'estatus_prod', 'limite_credito', 'saldo_actual', 'utilizacion_pct', 'tasa_interes_anual', 'edad', 'sexo', 'estado', 'ciudad', 'nivel_educativo', 'ocupacion', 'ingreso_mensual_mxn', 'antiguedad_dias', 'es_hey_pro', 'nomina_domiciliada', 'canal_apertura', 'score_buro', 'dias_desde_ultimo_login', 'preferencia_canal', 'satisfaccion_1_10', 'recibe_remesas', 'usa_hey_shop', 'idioma_preferido', 'tiene_seguro', 'num_productos_activos', 'patron_uso_atipico_cli']


,transaccion_id,user_id,producto_id,fecha_hora,tipo_operacion,canal,monto,comercio_nombre,categoria_mcc,ciudad_transaccion,estatus,motivo_no_procesada,intento_numero,meses_diferidos,cashback_generado,descripcion_libre,hora_del_dia,dia_semana,es_internacional,dispositivo,patron_uso_atipico_txn,es_dato_sintetico,user_id_prod,tipo_producto,estatus_prod,limite_credito,saldo_actual,utilizacion_pct,tasa_interes_anual,edad,sexo,estado,ciudad,nivel_educativo,ocupacion,ingreso_mensual_mxn,antiguedad_dias,es_hey_pro,nomina_domiciliada,canal_apertura,score_buro,dias_desde_ultimo_login,preferencia_canal,satisfaccion_1_10,recibe_remesas,usa_hey_shop,idioma_preferido,tiene_seguro,num_productos_activos,patron_uso_atipico_cli
0,TXN-0000000055,USR-00001,PRD-00000002,2025-01-15 14:17:42,compra,app_ios,33.88,DivertidoPark,entretenimiento,"Nueva York, NY",completada,NaN,1,NaN,0.34,Cargo automático,14,Wednesday,True,app_ios,False,True,USR-00001,tarjeta_credito_hey,activo,"144,000.00","88,790.40",0.62,35.71,21,M,Ciudad de México,CDMX - Benito Juárez,Preparatoria,Empleado,24500,1554,True,False,App,527,1,app_android,10.00,False,True,es_MX,False,2,False
1,TXN-0000000048,USR-00001,PRD-00000001,2025-01-17 00:31:56,cargo_recurrente,app_ios,249.00,GamerPass,servicios_digitales,CDMX - Benito Juárez,completada,NaN,1,NaN,NaN,Cargo automático,0,Friday,False,app_ios,False,True,USR-00001,cuenta_debito,activo,NaN,"80,954.60",NaN,NaN,21,M,Ciudad de México,CDMX - Benito Juárez,Preparatoria,Empleado,24500,1554,True,False,App,527,1,app_android,10.00,False,True,es_MX,False,2,False
2,TXN-0000000018,USR-00001,PRD-00000001,2025-01-17 22:48:23,cargo_recurrente,app_huawei,399.00,CloudDrive MX,servicios_digitales,CDMX - Benito Juárez,completada,NaN,1,NaN,NaN,inv hey,22,Friday,False,app_huawei,False,True,USR-00001,cuenta_debito,activo,NaN,"80,954.60",NaN,NaN,21,M,Ciudad de México,CDMX - Benito Juárez,Preparatoria,Empleado,24500,1554,True,False,App,527,1,app_android,10.00,False,True,es_MX,False,2,False


### 7d. Resumen: cobertura de joins

In [13]:
resumen_joins = pd.DataFrame([
    {
        "Join": "clientes ← productos (user_id, left)",
        "Filas resultado": clientes_productos.shape[0],
        "Columnas resultado": clientes_productos.shape[1],
        "Nulos en user_id": clientes_productos["user_id"].isnull().sum(),
    },
    {
        "Join": "transacciones ← productos (producto_id, inner)",
        "Filas resultado": txn_productos.shape[0],
        "Columnas resultado": txn_productos.shape[1],
        "Nulos en user_id": txn_productos["user_id_txn"].isnull().sum(),
    },
    {
        "Join": "transacciones ← productos ← clientes (completo)",
        "Filas resultado": df_full.shape[0],
        "Columnas resultado": df_full.shape[1],
        "Nulos en user_id": df_full["user_id"].isnull().sum(),
    },
])

display(resumen_joins)

,Join,Filas resultado,Columnas resultado,Nulos en user_id
0,"clientes ← productos (user_id, left)",38909,34,0
1,"transacciones ← productos (producto_id, inner)",802384,28,0
2,transacciones ← productos ← clientes (completo),802384,50,0
